# M2: Baseline Generation on Colab

在 Colab T4 GPU 上跑 `eval/generate_baseline.py`，让 **未微调、4-bit 量化的 Llama-2-7B** 对 50 道 eval 题生成 completions。

产物：`data/processed/baseline_generations.jsonl`。回本地后再用 `eval/scoring.py` 算 pass@1，得到 M4 的对照基线。

> **精度设计**：baseline 用 4-bit 量化（不是 fp16），是为了跟 M3 QLoRA 训练精度一致，避免 M2→M4 的改进被"量化 vs 没量化"混淆。详见 `CLAUDE.md`。

## 前置准备清单

**开始跑之前，把下面每一项都勾掉。**

- [ ] **代码已 push 到 GitHub** — Colab 从远程 clone，本地未 push 的改动跑的不是最新版
  - `git push origin dev/t`（或你当前的分支名）
- [x] **HuggingFace 账号 + Read Token** — https://huggingface.co/settings/tokens
  - 权限选 `Read` 即可，创建后**立刻复制保存**（HF 只显示一次）
- [x] **Llama 2 授权已批准** — https://huggingface.co/meta-llama/Llama-2-7b-hf 页面点 "Request access"
  - 用**同一个 HF 邮箱**（跟 token 那个账号）
  - Meta 通常几分钟到几小时批，没批准之前 `from_pretrained` 会 403
- [ ] **Colab Runtime 选 T4 GPU** — 顶部菜单 Runtime → Change runtime type → Hardware accelerator = T4 GPU
  - 免费档就有 T4，15GB VRAM 足够跑 4-bit 7B
- [x] 【改成public】**（如果仓库是 private）GitHub Personal Access Token** — Settings → Developer settings → Personal access tokens (classic)，勾 `repo` scope

**打开这个 notebook 的方式**：Colab → File → Open notebook → GitHub tab → 粘贴仓库 URL → 选 `notebooks/run_baseline_colab.ipynb`。这样每次打开都是仓库里的最新版，不会跟本地脱节。

## 关键细节（跑之前扫一眼）

- **VRAM 预算**：T4 有 15GB，4-bit Llama-2-7B 约 4-5GB，很宽裕
- **模型首次下载**：约 13GB 权重，Colab 内网 1-2 分钟；同一 runtime 内不会重复下载
- **生成耗时**：50 题 × greedy decode（每题最多 256 tokens），预计 5-10 分钟
- **会话超时**：免费档 12 小时硬上限、90 分钟空闲断连——不要开着页面走开吃饭
- **确定性**：脚本里写死了 `do_sample=False`（greedy），相同 prompt 每次输出一样，符合 pass@1 标准做法
- **停止规则**：脚本用 `STOP_SEQUENCES`（`\ndef `、`\nclass ` 等）截断生成，防止模型跑飞把多个函数拼一起。如果观察到 completion 被截得太狠或截得不够，去 `eval/generate_baseline.py` 调 `STOP_SEQUENCES`。
- **别改脚本**：这个 notebook 只是启动器，脚本原封不动跑。要改逻辑就在本地改、commit、push、然后重新在 Colab clone。

## Step 0: 确认 GPU

如果输出显示 `Tesla T4` 就 OK。如果报错 "command not found" 或没 GPU 信息 → runtime 没选对，回去改 Runtime type。

In [ ]:
!nvidia-smi

## Step 1: Clone 仓库

**改成你自己的 GitHub 用户名和当前分支名**，然后跑。

In [ ]:
GITHUB_USER = "siyux1927"   # ← 改
BRANCH = "dev/t"                # ← 改（或者 main）
REPO = "code-llm-finetuning"

# public repo:
!git clone -b {BRANCH} https://github.com/siyux1927/code-llm-finetuning.git

# private repo（把上一行注释掉，把下面这行的 YOUR_PAT 换成你的 GitHub PAT）:
# !git clone -b {BRANCH} https://YOUR_PAT@github.com/{GITHUB_USER}/{REPO}.git

%cd {REPO}
!ls data/processed/  # 应该看到 train_50.jsonl 和 eval_50.jsonl

## Step 2: 装依赖

`requirements-colab.txt` 是 GPU 专属依赖（`transformers`、`accelerate`、`bitsandbytes`），Mac 本地跑不了所以单独一份。

In [ ]:
!pip install -q -r requirements.txt -r requirements-colab.txt

## Step 3: 登录 HuggingFace

运行下面 cell，弹出输入框时**粘贴你的 HF Read Token**。

如果 Step 4 报 `401` / `403` / `gated model`：
- token 无效 → 回 HF settings 重新生成
- Llama 2 授权没批 → 去 https://huggingface.co/meta-llama/Llama-2-7b-hf 检查状态

In [ ]:
from huggingface_hub import login
login()

## Step 4: 跑 baseline 生成

预计 5-10 分钟。会看到每一题打印 `HumanEval/xx: generated N chars`。

如果第一题就卡很久没输出：多半是模型还在下载（13GB），耐心等 1-2 分钟第一题才会 print。

In [ ]:
!python eval/generate_baseline.py \
    --eval-set data/processed/eval_50.jsonl \
    --output data/processed/baseline_generations.jsonl

## Step 5: 快速 sanity check

看一眼输出文件对不对：50 行、每行有 `task_id` 和 `completion`、`completion` 非空。

In [ ]:
import json
from pathlib import Path

out = Path("data/processed/baseline_generations.jsonl")
lines = out.read_text().strip().split("\n")
print(f"总行数: {len(lines)}")

empty = 0
for line in lines:
    row = json.loads(line)
    if not row["completion"].strip():
        empty += 1
print(f"空 completion 数: {empty}")

print("\n--- 前 3 条样例 ---")
for line in lines[:3]:
    row = json.loads(line)
    print(f"[{row['task_id']}]")
    print(row['completion'][:200])
    print()

## Step 6: 把结果拿回本地

两种方式，二选一。

### 方式 A（推荐）：直接下载文件

文件很小（几十 KB），下完手动放回本地仓库的 `data/processed/`。

In [ ]:
from google.colab import files
files.download("data/processed/baseline_generations.jsonl")

### 方式 B：commit + push 回 GitHub

省事但要配 git 身份 + GitHub PAT。push 时提示密码就粘 PAT（不是 GitHub 登录密码）。

**注意**：如果本地那台机器同一分支也有未 push 的改动，会冲突——push 前先确认本地是干净的。

In [ ]:
!git config user.email "siyux1927@proton.me"    # ← 改
!git config user.name "siyux1927"                  # ← 改

!git add data/processed/baseline_generations.jsonl
!git commit -m "M2: 添加 baseline 生成结果"
!git push origin {BRANCH}

## 回到本地：算 pass@1

`eval/scoring.py` 目前只是个 library（暴露 `score_completions`），没有 CLI 入口。回本地后需要写一个小的入口脚本或者直接在 REPL 里跑，大致这样：

```python
import json
from pathlib import Path
from eval.scoring import score_completions

eval_set = {json.loads(l)["task_id"]: json.loads(l)
            for l in Path("data/processed/eval_50.jsonl").read_text().splitlines()}
gens = [json.loads(l) for l in Path("data/processed/baseline_generations.jsonl").read_text().splitlines()]

records = [{
    "task_id": g["task_id"],
    "prompt": eval_set[g["task_id"]]["prompt"],
    "completion": g["completion"],
    "test": eval_set[g["task_id"]]["test"],
    "entry_point": eval_set[g["task_id"]]["entry_point"],
} for g in gens]

result = score_completions(records)
print(f"pass@1 = {result['pass_at_1']:.3f} ({result['num_passed']}/{result['num_problems']})")
```

这个 baseline 数字就是后续 M4 fine-tuned pass@1 的对照基线，记到 CLAUDE.md 里。